# 06 · Limpieza · demanda comercial por CIIU (XM)

La demanda comercial no regulada, desagregada por actividad económica (CIIU):
**17,5 millones de filas**, una por hora y por subactividad, con 367
combinaciones `(Activity, Subactivity)` a lo largo del histórico y entre 349 y
355 activas en cada hora.

Lo que se encontró al mirarla:

- **90 302 valores vacíos** dentro de filas que sí existen;
- sectores que aparecen y desaparecen: alguno vive solo unas semanas;
- ni ceros ni negativos.

**Por qué por grupos.** `limpiar()` está pensada para una serie con una fila por
hora, y se niega a tratar una tabla así: deduplicar por hora borraría el 99 % de
los datos. Hay que partirla en 367 series y limpiar cada una por separado, para
que los estadísticos de un sector no contaminen a otro —10 MWh es normal en la
industria y disparatado en una biblioteca—. Lo hace
`limpieza.grupos.limpiar_por_grupos()`.

La rejilla horaria de cada sector va de **su** primera a **su** última hora: a
un sector que solo existió unos meses no se le inventan horas antes ni después.

In [1]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

import json, time
import pyarrow as pa
import pyarrow.parquet as pq

raiz     : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER
datasets : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets -> existe


## Carga con memoria acotada

Leído tal cual, el CSV ocupa más de 1,6 GB en memoria por las descripciones de
actividad repetidas en cada fila. Se lee por trozos, con `Activity` y
`Subactivity` como categorías y la fecha ya convertida.

In [2]:
from pandas.api.types import union_categoricals

t0 = time.time()
partes = []
for ruta in sorted(p for p in DATASETS.glob("xm/ciiu/*.csv.gz")):
    for trozo in pd.read_csv(ruta, usecols=["timestamp", "valor", "Activity", "Subactivity"], chunksize=2_000_000):
        trozo["timestamp"] = pd.to_datetime(trozo["timestamp"], format="ISO8601")
        trozo["Activity"] = trozo["Activity"].astype("category")
        trozo["Subactivity"] = trozo["Subactivity"].astype("category")
        partes.append(trozo)

ciiu = pd.DataFrame({
    "timestamp": pd.concat([p.timestamp for p in partes], ignore_index=True),
    "valor": pd.concat([p.valor for p in partes], ignore_index=True),
    "Activity": union_categoricals([p.Activity for p in partes]),
    "Subactivity": union_categoricals([p.Subactivity for p in partes]),
})
del partes
print(f"{len(ciiu):,} filas en {time.time() - t0:.0f} s · {ciiu.memory_usage(deep=True).sum() / 1e6:.0f} MB en memoria")

17,536,680 filas en 44 s · 333 MB en memoria


## Diagnóstico

In [3]:
g = ciiu.groupby(["Activity", "Subactivity"], observed=True)
filas = g.size()
vida = g.timestamp.agg(["min", "max"])
esperadas = ((vida["max"] - vida["min"]) / pd.Timedelta(hours=1) + 1).astype(int)
nulos = g.valor.apply(lambda s: int(s.isna().sum()))

print(f"actividades                   : {ciiu.Activity.nunique()}")
print(f"grupos (actividad, subactividad): {len(filas)}")
print(f"filas por grupo               : min {filas.min():,} · mediana {int(filas.median()):,} · máx {filas.max():,}")
print(f"grupos con horas sin fila     : {int((esperadas > filas).sum())} ({int((esperadas - filas).sum()):,} horas)")
print(f"valores vacíos                : {int(ciiu.valor.isna().sum()):,} en {int((nulos > 0).sum())} grupos")
print(f"ceros / negativos             : {int((ciiu.valor == 0).sum())} / {int((ciiu.valor < 0).sum())}")
por_hora = ciiu.groupby("timestamp").size()
print(f"subactividades por hora       : {por_hora.min()} .. {por_hora.max()}")

actividades                   : 21
grupos (actividad, subactividad): 367
filas por grupo               : min 984 · mediana 49,776 · máx 49,824
grupos con horas sin fila     : 16 (95,304 horas)
valores vacíos                : 90,302 en 111 grupos
ceros / negativos             : 0 / 0


subactividades por hora       : 1 .. 356


In [4]:
print("Los sectores de vida más corta:")
vida.assign(filas=filas).sort_values("filas").head(6)

Los sectores de vida más corta:


,,min,max,filas
Activity,Subactivity,,,
TRANSPORTE Y ALMACENAMIENTO,ACTIVIDADES POSTALES NACIONALES,2021-01-01 00:00:00-05:00,2021-02-10 23:00:00-05:00,984
INDUSTRIAS MANUFACTURERAS,FABRICACIÓN DE APARATOS ELECTRÓNICOS DE CONSUMO,2021-01-01 00:00:00-05:00,2021-04-07 23:00:00-05:00,2328
EDUCACIÓN,ACTIVIDADES DE APOYO A LA EDUCACIÓN,2023-06-15 00:00:00-05:00,2023-12-31 23:00:00-05:00,4800
ACTIVIDADES DE LOS HOGARES INDIVIDUALES EN CALIDAD DE EMPLEADORES; ACTIVIDADES NO DIFERENCIADAS DE LOS HOGARES INDIVIDUALES COMO PRODUCTORES DE BIENES Y SERVICIOS PARA USO PROPIO,ACTIVIDADES NO DIFERENCIADAS DE LOS HOGARES INDIVIDUALES COMO PRODUCTORES DE BIENES PARA USO PROPIO,2026-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,5952
COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS AUTOMOTORES Y MOTOCICLETAS,"COMERCIO AL POR MENOR DE COMPUTADORES, EQUIPOS PERIFÉRICOS, PROGRAMAS DE INFORMÁTICA Y EQUIPOS DE TELECOMUNICACIONES EN ESTABLECIMIENTOS ESPECIALIZADOS",2025-11-13 00:00:00-05:00,2026-09-05 23:00:00-05:00,7128
ALOJAMIENTO Y SERVICIOS DE COMIDA,EXPENDIO DE BEBIDAS ALCOHÓLICAS PARA EL CONSUMO DENTRO DEL ESTABLECIMIENTO,2021-01-01 00:00:00-05:00,2022-01-31 23:00:00-05:00,9504


## Limpieza por grupos

Cada sector se limpia como una serie propia y se escribe en cuanto está listo,
sin juntar el resultado en memoria. El Parquet final se escribe primero como
`.tmp` y se renombra al acabar, para que una ejecución interrumpida nunca deje
un archivo que parezca completo.

Tarda unos minutos: son 367 series de hasta 49 824 horas.

In [5]:
from limpieza.grupos import limpiar_por_grupos, fila_de_resumen, resumir

SALIDA = DATASETS / "limpios"
SALIDA.mkdir(parents=True, exist_ok=True)
destino = SALIDA / "ciiu_limpio.parquet"
temporal = destino.with_name(destino.name + ".tmp")

ESQUEMA = pa.schema([
    ("timestamp", pa.timestamp("ns", tz="America/Bogota")),
    ("activity", pa.string()), ("subactivity", pa.string()),
    ("valor", pa.float64()), ("origen_valor", pa.string()),
    ("imputado", pa.bool_()), ("hueco_horas", pa.int64()),
    ("z_estacional", pa.float64()), ("atipico", pa.bool_()),
    ("atipico_evaluable", pa.bool_()), ("atipico_iqr", pa.bool_()),
    ("atipico_global", pa.bool_()), ("atipico_iqr_global", pa.bool_()),
    ("periodo_atipico", pa.bool_()),
])

filas_resumen = []
t0 = time.time()
with pq.ParquetWriter(temporal, ESQUEMA, compression="zstd") as escritor:
    for n, (etiqueta, limpio, registro) in enumerate(limpiar_por_grupos(ciiu, ["Activity", "Subactivity"]), 1):
        assert not limpio.timestamp.duplicated().any(), f"horas repetidas en {etiqueta}"
        tabla = limpio[ESQUEMA.names].astype({"hueco_horas": "int64"})
        escritor.write_table(pa.Table.from_pandas(tabla, schema=ESQUEMA, preserve_index=False))
        filas_resumen.append(fila_de_resumen(etiqueta, limpio, registro))
        if n % 60 == 0:
            print(f"  {n:>3} grupos · {time.time() - t0:.0f} s")
temporal.replace(destino)

total = resumir(filas_resumen)
print(f"{total['n_grupos']} grupos limpiados en {time.time() - t0:.0f} s")
total

   60 grupos · 48 s


  120 grupos · 90 s


  180 grupos · 133 s


  240 grupos · 173 s


  300 grupos · 212 s


  360 grupos · 261 s


367 grupos limpiados en 266 s


{'n_grupos': 367,
 'filas_iniciales': 17536680,
 'filas_finales': 17631984,
 'horas_creadas': 95304,
 'observados': 17446378,
 'interpolados': 8511,
 'faltantes': 177095,
 'atipicos': 901528,
 'no_evaluables': 1401758,
 'grupos_con_interpolados': 102,
 'grupos_con_faltantes': 90,
 'pct_observado': 98.9473}

## Resultado

In [6]:
grupos = pd.DataFrame(filas_resumen)
print(f"observados  : {total['observados']:>12,}  ({total['pct_observado']} %)")
print(f"interpolados: {total['interpolados']:>12,}  en {total['grupos_con_interpolados']} grupos (huecos de hasta 3 h)")
print(f"faltantes   : {total['faltantes']:>12,}  en {total['grupos_con_faltantes']} grupos (huecos largos, se quedan NaN)")
print(f"atípicos    : {total['atipicos']:>12,}  (causales, dentro de cada sector)")
print(f"horas creadas para completar la rejilla de cada sector: {total['horas_creadas']:,}")
print()
print("Sectores con más horas sin valor:")
grupos.sort_values("faltantes", ascending=False).head(8)[
    ["Activity", "Subactivity", "desde", "hasta", "faltantes", "tramos_no_imputados", "interpolados"]
]

observados  :   17,446,378  (98.9473 %)
interpolados:        8,511  en 102 grupos (huecos de hasta 3 h)
faltantes   :      177,095  en 90 grupos (huecos largos, se quedan NaN)
atípicos    :      901,528  (causales, dentro de cada sector)
horas creadas para completar la rejilla de cada sector: 95,304

Sectores con más horas sin valor:


,Activity,Subactivity,desde,hasta,faltantes,tramos_no_imputados,interpolados
166,DISTRIBUCIÓN DE AGUA; EVACUACIÓN Y TRATAMIENTO...,ACTIVIDADES DE SANEAMIENTO AMBIENTAL Y OTRO SE...,2021-04-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,32923,930,1094
162,CONSTRUCCIÓN,INSTALACIONES ELÉCTRICAS,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,24888,2074,0
206,INDUSTRIAS MANUFACTURERAS,CONSTRUCCIÓN DE EMBARCACIONES DE RECREO Y DEPORTE,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,22959,17,28
232,INDUSTRIAS MANUFACTURERAS,FABRICACIÓN DE ARTÍCULOS DE PIEL,2021-01-01 00:00:00-05:00,2026-06-14 23:00:00-05:00,22631,10,0
30,ACTIVIDADES DE SERVICIOS ADMINISTRATIVOS Y DE ...,ACTIVIDADES DE SERVICIOS DE SISTEMAS DE SEGURIDAD,2021-01-01 00:00:00-05:00,2025-01-31 23:00:00-05:00,17928,1491,0
9,"ACTIVIDADES ARTÍSTICAS, DE ENTRETENEMIENTO Y R...",GESTIÓN DE INSTALACIONES DEPORTIVAS,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,13862,4,12
337,OTRAS ACTIVIDADES DE SERVICIOS,ACTIVIDADES DE ASOCIACIONES PROFESIONALES,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,11860,2,7
180,EDUCACIÓN,EDUCACIÓN TECNOLÓGICA,2021-01-01 00:00:00-05:00,2026-09-05 23:00:00-05:00,6067,794,633


## Comprobaciones

In [7]:
meta = pq.ParquetFile(destino).metadata
assert meta.num_rows == total["filas_finales"], "el Parquet no tiene las filas esperadas"
assert total["filas_iniciales"] == len(ciiu), "no se procesaron todas las filas de entrada"
assert total["observados"] + total["interpolados"] + total["faltantes"] == total["filas_finales"]
assert total["filas_finales"] - total["filas_iniciales"] == total["horas_creadas"]
print(f"OK: {len(ciiu):,} filas de entrada -> {meta.num_rows:,} en el Parquet")
print(f"    (+{total['horas_creadas']:,} horas que faltaban en la rejilla de su sector, marcadas)")

OK: 17,536,680 filas de entrada -> 17,631,984 en el Parquet
    (+95,304 horas que faltaban en la rejilla de su sector, marcadas)


## Registro

In [8]:
registro_ciiu = {
    "version_formato": 1,
    "conjunto": "ciiu_demacomenoreg",
    "momento": pd.Timestamp.now().isoformat(timespec="seconds"),
    "archivo": destino.name,
    "politica": ("limpieza por (Activity, Subactivity); rejilla de cada sector entre su primera "
                 "y su ultima hora; interpolacion solo en huecos de hasta 3 h; atipicos causales "
                 "dentro de cada sector; nada se elimina"),
    "resumen": total,
    "grupos": filas_resumen,
}
with open(SALIDA / "registro_ciiu.json", "w", encoding="utf-8") as f:
    json.dump(registro_ciiu, f, ensure_ascii=False, indent=2, default=str)
print("registro ->", SALIDA / "registro_ciiu.json")

registro -> C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets\limpios\registro_ciiu.json


## Cómo leerlo sin cargar las 17,5 M de filas

El Parquet se escribió sector a sector, así que se puede leer solo una parte.

In [9]:
educacion = pq.read_table(
    destino,
    columns=["timestamp", "subactivity", "valor", "origen_valor", "atipico"],
    filters=[("activity", "==", "EDUCACIÓN")],
).to_pandas()
print(f"EDUCACIÓN: {len(educacion):,} filas · {educacion.subactivity.nunique()} subactividades")
educacion.head()

EDUCACIÓN: 651,888 filas · 14 subactividades


,timestamp,subactivity,valor,origen_valor,atipico
0,2021-01-01 00:00:00-05:00,EDUCACIÓN BÁSICA PRIMARIA,146.67,observado,False
1,2021-01-01 01:00:00-05:00,EDUCACIÓN BÁSICA PRIMARIA,147.40,observado,False
2,2021-01-01 02:00:00-05:00,EDUCACIÓN BÁSICA PRIMARIA,149.00,observado,False
3,2021-01-01 03:00:00-05:00,EDUCACIÓN BÁSICA PRIMARIA,148.29,observado,False
4,2021-01-01 04:00:00-05:00,EDUCACIÓN BÁSICA PRIMARIA,143.31,observado,False
